# 📈 Módulo 07 - Notebook 01: Resampling y Ventanas Móviles

## 🕒 Series de Tiempo: Agregaciones Temporales y Promedios Móviles

**Libro:** Saliendo de lo Pandito  
**Módulo:** 07 - Series de Tiempo Financieras  
**Duración estimada:** 70 minutos  
**Dificultad:** 🟡 Intermedio  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** índices de tiempo con DatetimeIndex  
✅ **Aplicar** resampling (cambio de frecuencia)  
✅ **Calcular** ventanas móviles (rolling)  
✅ **Analizar** tendencias con promedios móviles  
✅ **Detectar** estacionalidad en series temporales

---

## 📋 Pre-requisitos

* ✅ Módulo 06 completado (Agregaciones)
* ✅ Conocimiento de fechas en Pandas (pd.to_datetime)
* ✅ Familiaridad con series temporales

---

## 📚 Contenido

1. DatetimeIndex y Series Temporales
2. Resampling: Cambio de Frecuencia
3. Ventanas Móviles (Rolling)
4. Suavizado de Series
5. Detección de Tendencias
6. Caso Integrador: Análisis de Ventas Mensuales

---

## 💡 Por qué importa

**Series de tiempo están en todas partes:**

* 💰 **Finanzas:** Precios de acciones, tipos de cambio
* 📊 **Ventas:** Ingresos mensuales, estacionalidad
* 🏭 **Economía:** PIB, inflación, desempleo
* 🏪 **Retail:** Tráfico, inventario, demanda

**Dominar series de tiempo = Predecir el futuro con datos del pasado**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df_raw = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_raw['fecha'] = pd.to_datetime(df_raw['fecha'])
    
    # Opción 1: Serie temporal agregada (todas las sucursales)
    df_ts = df_raw.groupby('fecha')['ventas'].sum().sort_index()
    df_ts = df_ts.to_frame(name='ventas_totales')
    
    # Opción 2: Serie por sucursal
    df_ts_sucursal = df_raw.pivot_table(
        values='ventas', 
        index='fecha', 
        columns='sucursal_nombre', 
        aggfunc='sum'
    )
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📅 Período: {df_ts.index.min().strftime('%Y-%m-%d')} a {df_ts.index.max().strftime('%Y-%m-%d')}")
    print(f"   📊 Meses totales: {len(df_ts)}")
    print(f"   🏪 Sucursales: {df_raw['sucursal_id'].nunique()}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📋 DataFrames disponibles:")
    print(f"   • df_ts: Serie temporal agregada (todas las sucursales)")
    print(f"   • df_ts_sucursal: Serie temporal por sucursal")
    print(f"   • df_raw: DataFrame completo con todas las columnas")
    
    print(f"\n🎯 Este notebook usará datos REALES de Los Andes Market")
    print(f"   Los ejemplos trabajarán con series temporales reales")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ts = None
    df_ts_sucursal = None
    df_raw = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Series de Tiempo: Conceptos Fundamentales

### 🕒 ¿Qué es una Serie de Tiempo?

Una **serie de tiempo** es una secuencia de datos ordenados cronológicamente.

**Ejemplo:**
```
Fecha         Ventas
2023-01      100,000
2023-02      125,000
2023-03      115,000
```

**Componentes:**
1. **Índice temporal:** Fechas en orden
2. **Valores:** Métrica que evoluciona en el tiempo

---

### 🗓️ DatetimeIndex

**DatetimeIndex** es el índice especializado de Pandas para series de tiempo.

```python
# Crear DatetimeIndex
fechas = pd.date_range(start='2023-01-01', periods=12, freq='MS')  # Monthly Start
df = pd.DataFrame({'ventas': [...]}, index=fechas)
```

**Ventajas:**
* Operaciones temporales eficientes
* Resample automático
* Slicing por fechas

---

### 🔄 Resampling (Cambio de Frecuencia)

**Resampling** = Cambiar la frecuencia de la serie.

#### 1️⃣ **Downsampling** (Mayor → Menor frecuencia)

```python
# Diario → Mensual
df_mensual = df_diario.resample('M').sum()
```

**Ejemplo:**
```
Diario:         Mensual:
2023-01-01  10  2023-01  310  (suma de 31 días)
2023-01-02  12  2023-02  280  (suma de 28 días)
...
2023-01-31   8
```

---

#### 2️⃣ **Upsampling** (Menor → Mayor frecuencia)

```python
# Mensual → Diario
df_diario = df_mensual.resample('D').ffill()  # forward fill
```

**Métodos de rellenado:**
* `ffill()`: Forward fill (repetir último valor)
* `bfill()`: Backward fill (usar próximo valor)
* `interpolate()`: Interpolación lineal

---

### 📊 Ventanas Móviles (Rolling)

**Rolling window** = Calcular una métrica sobre una ventana deslizante.

```python
# Promedio móvil de 3 meses
df['MA_3M'] = df['ventas'].rolling(window=3).mean()
```

**Visualización:**
```
Mes    Ventas  MA_3M
01     100     NaN      (sólo 1 valor)
02     120     NaN      (sólo 2 valores)
03     110     110.0    (promedio de 100, 120, 110)
04     130     120.0    (promedio de 120, 110, 130)
05     140     126.7    (promedio de 110, 130, 140)
```

---

### 🎯 Funciones de Agregación

**En resample():**
```python
df.resample('M').sum()    # Suma mensual
df.resample('M').mean()   # Promedio mensual
df.resample('M').max()    # Máximo mensual
```

**En rolling():**
```python
df['ventas'].rolling(3).mean()   # Promedio móvil
df['ventas'].rolling(3).std()    # Desviación estándar móvil
df['ventas'].rolling(3).sum()    # Suma móvil
```

---

### 💼 Casos de Uso

| Operación | Uso típico |
|-----------|---------------|
| **Resample diario → mensual** | Reportes mensuales de ventas diarias |
| **Rolling mean (3M)** | Suavizar volatilidad, detectar tendencia |
| **Rolling std** | Medir volatilidad en finanzas |
| **Resample + ffill** | Rellenar datos faltantes |

---

### 💡 Regla de Oro

👉 **Resample** = Cambiar frecuencia  
👉 **Rolling** = Ventana deslizante para suavizar/calcular

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🕒 SERIES DE TIEMPO: RESAMPLING Y ROLLING")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

print("\n🎯 En este notebook aprenderás:")
print("  • DatetimeIndex: Índices temporales")
print("  • Resampling: Cambio de frecuencia (diario → mensual)")
print("  • Rolling: Ventanas móviles (promedios móviles)")
print("  • Suavizado de series temporales")

print("\n📖 Métodos clave:")
print("  - pd.date_range(start, periods, freq)")
print("  - df.resample('M').sum()  # Cambiar frecuencia")
print("  - df['col'].rolling(window=3).mean()  # Ventana móvil")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")